In [ ]:
config = {
    "vessel_shape": (4,5,5),
    "yard_shape": (4, 5, 5),
    "num_containers": 100,
    "group_num": 3,
    "num_cranes": 5, 
    "group_placement": "random",
    "seed": 4307,
    "time_penalty_coef": 0.002
}

In [ ]:
from envs.stowage_crane_gym import MultiCraneStowageEnv
env = MultiCraneStowageEnv(config=config, render_mode="rgb_array")
obs, info = env.reset()

In [ ]:
import matplotlib.pyplot as plt

# Get the RGB array
rgb_array = env.render()

# Display the image
plt.imshow(rgb_array)
plt.axis('off')  # Hide axes
plt.show()

In [ ]:
env.crane_busy_until

In [ ]:
obs, _,terminated,_,info = env.step(500)
print(info["cranes"]["busy_until"]-info["current_time"])
print(env.current_vessel_slots)
print(terminated)

In [ ]:
import numpy as np
for a in list(np.arange(env.action_space.n)[env.action_masks()]):
    print(a, env._decode_action(a))

In [ ]:
from envs.stowage_crane_gym import MultiCraneStowageEnv
from sb3_contrib import MaskablePPO
from algorithms.DQN import MaskableDQN
from algorithms.A2C import MaskableA2C
from sb3_contrib.common.maskable.utils import get_action_masks
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3 import PPO
import numpy as np


class TensorboardCallback(BaseCallback):
    """
    Custom callback for plotting additional values in tensorboard.
    """

    def __init__(self, verbose=0):
        self.is_tb_set = False
        super(TensorboardCallback, self).__init__(verbose)

    def _on_step(self) -> bool:
        if self.num_timesteps % 5000 == 0:
        #     print(self.locals.keys())
            # Evaluate the policy
            env = MultiCraneStowageEnv(config=config)
            obs, info = env.reset()
            rewards = []
            while True:
                action, _ = self.model.predict(obs, action_masks=env.action_masks())
                obs, reward, terminated, truncated, info = env.step(action)
                if terminated or truncated:
                    rewards.append(reward)
                    total_shifters = info.get("total_shifters")
                    current_time = info.get("current_time")
                    break
            mean_r = np.mean(rewards)
            # Log to tensorboard
            self.logger.record("metrics/mean_reward", mean_r)
            self.logger.record("metrics/total_shifters", total_shifters)
            self.logger.record("metrics/current_time", current_time)
            self.logger.dump(self.num_timesteps)
        return True

env = MultiCraneStowageEnv(config=config)
obs, info = env.reset()

# model = MaskableDQN("MlpPolicy", env, verbose=1, gamma=0.4)
# model.learn(100, callback=TensorboardCallback(), log_interval=4)
# env.reset()
model = MaskablePPO("MlpPolicy", env, gamma=0.4, verbose=1,tensorboard_log="./multi_cranes/", n_steps=2000)
model.learn(80000,callback=TensorboardCallback(),tb_log_name="PPO_100c_3g_5cr")

In [ ]:
obs, _ = env.reset()
while True:
    action, _ = model.predict(obs, action_masks=get_action_masks(env))
    obs, reward, terminated, truncated, info = env.step(action)
    if terminated or truncated:
        break